# Preparación de los corpus

Este notebook convierte los tres corpus crudos (A = PROMISE, B = PROMISE_exp, D = PURE), tal como se descargaron de sus fuentes, en un solo archivo homogéneo.

Entrada: `datasets/` (archivos crudos, que no se modifican). Salida: `datasets/harmonized/corpus_all.csv` con columnas `corpus, group, req_id, text, label_binary, label_original`.

El proceso es determinista: no hay aleatoriedad, de modo que dos ejecuciones producen bytes idénticos. La última celda verifica el SHA-256 del archivo generado contra la huella registrada (`3dd784dc...`), porque la muestra PURE-300 y todos los experimentos dependen de este CSV.

## Utilerías de limpieza

**Corrección de encoding** (`fix_encoding`, `CP1252_ESCAPES`). PROMISE_exp arrastra escapes RTF/Windows-1252; por ejemplo, `CampusConnect\92s` en lugar de `CampusConnect's`. Sin corregirlos, dos textos idénticos parecen distintos y la deduplicación falla: con la corrección, los duplicados detectados suben de 584 a 621. Se agrega la normalización Unicode NFKC para unificar variantes de un mismo carácter.

**Colapso de espacios** (`squash_whitespace`). Saltos de línea y espacios múltiples se reducen a un espacio, para que el mismo texto con distinto formato no cuente como distinto.

**Clave de deduplicación** (`dedup_key`). Minúsculas, sin puntuación, espacios colapsados. Se usa únicamente para comparar textos; el texto guardado conserva mayúsculas y puntuación, porque cada modelo aplica su propia representación sobre el texto. Por la misma razón no se eliminan stopwords ni se lematiza.

**Conteo de tokens alfabéticos** (`alphabetic_tokens`). Se usa para descartar fragmentos no clasificables en PURE.

In [1]:
import csv
import glob
import os
import re
import unicodedata
import xml.etree.ElementTree as ET
from collections import Counter

DATA = os.path.join("..", "datasets")
OUT_DIR = os.path.join(DATA, "harmonized")
os.makedirs(OUT_DIR, exist_ok=True)

CP1252_ESCAPES = {
    r"\91": "'", r"\92": "'", r"\93": '"', r"\94": '"',
    r"\95": "-", r"\96": "-", r"\97": "-", r"\85": "...",
}

def fix_encoding(text):
    for escape, replacement in CP1252_ESCAPES.items():
        text = text.replace(escape, replacement)
    return unicodedata.normalize("NFKC", text)

def squash_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

def dedup_key(text):
    return re.sub(r"[^a-z0-9 ]", "", squash_whitespace(text).lower()).strip()

def alphabetic_tokens(text):
    return len(re.findall(r"[a-zA-Z]+", text))

records = []

## Corpus A: PROMISE (625 requisitos, 15 proyectos)

Fuente: `promise_nfr.csv` del paquete de replicación de NoRBERT (Hey et al., 2020), el mismo archivo que usó NoRBERT.

`group` es el ProjectID, porque el proyecto es la unidad de partición del modo sin fuga. En `label_binary`, la clase F queda como F y las once subclases de NFR (A, FT, L, LF, MN, O, PE, SC, SE, US, PO) se agrupan como NFR; la subclase se conserva en `label_original`.

In [2]:
promise_path = os.path.join(
    DATA, "norbert-replication", "Code",
    "Task1_to_3_original_Promise_NFR_dataset", "promise_nfr.csv")

with open(promise_path, encoding="utf-8-sig") as f:
    for i, row in enumerate(csv.DictReader(f, delimiter=";")):
        subclass = row["class"].strip()
        records.append({
            "corpus": "PROMISE",
            "group": f"P{row['ProjectID'].strip()}",
            "req_id": f"A-{i+1:04d}",
            "text": squash_whitespace(fix_encoding(row["RequirementText"])),
            "label_binary": "F" if subclass == "F" else "NFR",
            "label_original": subclass,
        })

promise_keys = {dedup_key(r["text"]) for r in records}
print(f"Corpus A (PROMISE): {sum(r['corpus'] == 'PROMISE' for r in records)} requisitos")

Corpus A (PROMISE): 625 requisitos


## Corpus B: PROMISE_exp, solo los requisitos nuevos

PROMISE_exp (Lima et al., 2019) es una expansión que contiene a PROMISE. Evaluar A contra B sin deduplicar equivaldría a probar el modelo con sus propios datos de entrenamiento, lo que constituye fuga por duplicados (Kapoor y Narayanan, 2023).

Regla: todo requisito de PROMISE_exp cuya clave normalizada ya exista en A se descarta. Resultado: 621 duplicados eliminados y B queda con 348 requisitos nuevos.

In [3]:
dropped_promise_exp_duplicates = 0

with open(os.path.join(DATA, "promise_exp.csv"), encoding="utf-8-sig") as f:
    for i, row in enumerate(csv.DictReader(f)):
        text = squash_whitespace(fix_encoding(row["RequirementText"]))
        if dedup_key(text) in promise_keys:
            dropped_promise_exp_duplicates += 1
            continue
        subclass = row["_class_"].strip()
        records.append({
            "corpus": "PROMISE_exp_new",
            "group": f"X{row['ProjectID'].strip()}",
            "req_id": f"B-{i+1:04d}",
            "text": text,
            "label_binary": "F" if subclass == "F" else "NFR",
            "label_original": subclass,
        })

print(f"Corpus B: {sum(r['corpus'] == 'PROMISE_exp_new' for r in records)} nuevos | "
      f"{dropped_promise_exp_duplicates} duplicados de A descartados")

Corpus B: 348 nuevos | 621 duplicados de A descartados


## Corpus D: PURE (documentos de requisitos sin etiquetas)

Fuente: los XML de PURE (Ferrari et al., 2017) que traen requisitos marcados con el elemento `req`. El campo `label_binary` queda vacío porque PURE no trae etiquetas F/NFR; el estándar PURE-300 se construye aparte, con el protocolo de anotación.

Limpieza específica:

- Numeración inicial pegada (`LEADING_NUMBERING`), por ejemplo "5.2.3.41 It should...". Es un artefacto del documento, no contenido del requisito.
- Marcadores de obligatoriedad de EIRENE (`EIRENE_OBLIGATION_MARKERS`): (M), (O), (I), (MI), es decir mandatorio, opcional, informativo. Son metadatos del estándar, no lenguaje del requisito.
- Fragmentos de menos de `MIN_CLASSIFIABLE_TOKENS` (4) tokens: restos de tablas y títulos sin contenido clasificable.
- Duplicados exactos dentro del mismo documento.

Hay artefactos que esta versión no limpia: el elemento `modifier` de ERTMS que queda como letra M u O suelta al final, tablas aplanadas dentro del texto (EIRENE) y títulos de sección pegados al requisito anterior. Se dejan a propósito, porque la muestra de anotación se entregó con estos textos y los modelos deben ver exactamente lo mismo que los anotadores.

In [4]:
LEADING_NUMBERING = re.compile(r"^\s*\d+(\.\d+)*\.?\s+")
EIRENE_OBLIGATION_MARKERS = re.compile(r"\((?:M|O|I|MI)\)")
MIN_CLASSIFIABLE_TOKENS = 4

def clean_pure(text):
    text = LEADING_NUMBERING.sub("", text)
    text = EIRENE_OBLIGATION_MARKERS.sub(" ", text)
    return squash_whitespace(text)

dropped_pure_short = 0
dropped_pure_intradoc_duplicates = 0

for path in sorted(glob.glob(os.path.join(DATA, "pure-xml", "XMLZIPFile", "*.xml"))):
    document = os.path.basename(path).replace(".xml", "")
    try:
        root = ET.parse(path).getroot()
    except ET.ParseError:
        continue
    reqs = [e for e in root.iter() if e.tag.endswith("}req") or e.tag == "req"]
    seen_in_document = set()
    for i, req in enumerate(reqs):
        text = clean_pure(fix_encoding(squash_whitespace(" ".join(req.itertext()))))
        if alphabetic_tokens(text) < MIN_CLASSIFIABLE_TOKENS:
            dropped_pure_short += 1
            continue
        key = dedup_key(text)
        if key in seen_in_document:
            dropped_pure_intradoc_duplicates += 1
            continue
        seen_in_document.add(key)
        records.append({
            "corpus": "PURE",
            "group": document,
            "req_id": f"D-{document[:12].replace(' ', '_')}-{i+1:04d}",
            "text": text,
            "label_binary": "",
            "label_original": "",
        })

print(f"Corpus D (PURE): {sum(r['corpus'] == 'PURE' for r in records)} | "
      f"<{MIN_CLASSIFIABLE_TOKENS} tokens: {dropped_pure_short} | "
      f"dups intra-doc: {dropped_pure_intradoc_duplicates}")

Corpus D (PURE): 958 | <4 tokens: 9 | dups intra-doc: 37


## Salida, resumen y verificaciones de integridad

Las aserciones codifican lo que el protocolo promete: 625 requisitos en A, exactamente 621 duplicados detectados en B, cero escapes cp1252 y cero numeración residual en PURE. La última verificación es la garantía de congelamiento: el SHA-256 del CSV generado debe coincidir con la huella registrada; si no coincide, algo cambió y el archivo no puede usarse sin una enmienda documentada.

In [5]:
out_path = os.path.join(OUT_DIR, "corpus_all.csv")
with open(out_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "corpus", "group", "req_id", "text", "label_binary", "label_original"])
    writer.writeheader()
    writer.writerows(records)

from tabulate import tabulate

by_corpus = Counter(r["corpus"] for r in records)
resumen = []
for corpus in sorted(by_corpus):
    subset = [r for r in records if r["corpus"] == corpus]
    labels = Counter(r["label_binary"] for r in subset)
    resumen.append({
        "corpus": corpus,
        "requisitos": len(subset),
        "grupos": len({r["group"] for r in subset}),
        "etiquetas": ", ".join(f"{k or 'sin etiqueta'}: {v}" for k, v in sorted(labels.items())),
    })

print(f"Escrito: {out_path} ({len(records)} registros)\n")
print(tabulate(resumen, headers="keys", tablefmt="github"))

descartes = [
    ["PROMISE_exp duplicados de A", dropped_promise_exp_duplicates],
    [f"PURE fragmentos < {MIN_CLASSIFIABLE_TOKENS} tokens", dropped_pure_short],
    ["PURE duplicados intra-documento", dropped_pure_intradoc_duplicates],
]
print("\n" + tabulate(descartes, headers=["descarte", "n"], tablefmt="github"))

assert by_corpus["PROMISE"] == 625, "PROMISE debe tener 625"
assert dropped_promise_exp_duplicates == 621, "la deduplicación debe detectar exactamente 621 duplicados"
assert all("\\9" not in r["text"] for r in records), "quedan escapes cp1252"
assert not any(LEADING_NUMBERING.match(r["text"]) for r in records if r["corpus"] == "PURE"), \
    "queda numeración en PURE"

import hashlib
FROZEN_SHA256 = "3dd784dc1299f75133094efbdcb151eb5a6fbe47c97563449097e6419e3ebbea"
sha = hashlib.sha256(open(out_path, "rb").read()).hexdigest()
assert sha == FROZEN_SHA256, f"corpus_all.csv CAMBIÓ: {sha} != huella congelada"
print(f"\nVerificaciones de integridad: OK | SHA-256 coincide con la huella congelada ({sha[:16]}…)")

Escrito: ../datasets/harmonized/corpus_all.csv (1931 registros)

| corpus          |   requisitos |   grupos | etiquetas         |
|-----------------|--------------|----------|-------------------|
| PROMISE         |          625 |       15 | F: 255, NFR: 370  |
| PROMISE_exp_new |          348 |       34 | F: 191, NFR: 157  |
| PURE            |          958 |        6 | sin etiqueta: 958 |

| descarte                        |   n |
|---------------------------------|-----|
| PROMISE_exp duplicados de A     | 621 |
| PURE fragmentos < 4 tokens      |   9 |
| PURE duplicados intra-documento |  37 |

Verificaciones de integridad: OK | SHA-256 coincide con la huella congelada (3dd784dc1299f751…)
